# Preprocessing data

## Import data

In [1]:
from pathlib import Path

# Local data root (./data) instead of Google Drive.
# Works whether the kernel starts in encoder2/ or at the repo root.
ROOT = next((p for p in (Path("data"), Path("encoder2/data")) if p.is_dir()),
            Path("data")).resolve()
print(ROOT, ROOT.exists(), sorted(p.name for p in ROOT.glob("user*") if p.is_dir()))

/Users/albi/Documents/Projects/IBM_hackathon_2026/encoder2/data True ['user12', 'user15', 'user16', 'user20', 'user21', 'user23', 'user29', 'user35', 'user7', 'user9']


In [4]:
import pandas as pd

# Balabit session schema
# NOTE: timestamps are float64 on purpose. float32 holds ~7 significant digits;
# if either clock is a Unix epoch (~1.4e9) instead of session-relative seconds,
# float32 quantises it to ~100 s and every dt/velocity downstream is garbage.
DTYPES = {
    'record timestamp': 'float64',
    'client timestamp': 'float64',
    'button':           'category',
    'state':            'category',
    # float, not int: a single malformed or empty x/y cell makes an int dtype
    # raise and kills the whole read. float32 is exact for integers < 2**24.
    'x':                'float32',
    'y':                'float32',
}

def load_session(path):
    return pd.read_csv(path, dtype=DTYPES)

def user_key(p):
    # handles both 'user_7' (original Balabit layout) and 'user7' (./data layout)
    return int(''.join(c for c in p.name if c.isdigit()))

users = {}   # 'user7' -> {'session_0123456789': DataFrame}
for user_dir in sorted((p for p in ROOT.glob('user*') if p.is_dir()), key=user_key):
    sessions = {f.stem: load_session(f) for f in sorted(user_dir.glob('session_*'))}
    users[user_dir.name] = sessions
    print(f'{user_dir.name}: {len(sessions)} sessions, '
          f'{sum(len(d) for d in sessions.values()):,} events')

# flat list if you prefer positional access:
all_users = [list(users[u].values()) for u in users]        # list[list[DataFrame]]
all_sessions = [df for u in users.values() for df in u.values()]

ModuleNotFoundError: No module named 'pandas'

## Data preprocessing

raw event log -> per-stroke feature table.

Input CSV format (Balabit-style):
*    record timestamp, client timestamp,button,state,x,y

Pipeline:
*    load_events()      clean + dedupe + normalise the raw log
*    segment_strokes()  cut the move stream into strokes
*    strokes_to_frame() one row of ~55 features per stroke
*    strokes_to_windows()  (optional) aggregate N strokes -> one model input vector

Colab:
*    !pip -q install pandas numpy
*    from mouse_strokes import *
*    cfg = Config()
*    df  = process_file("session_0001", cfg, user="user7", session="session_0001")



In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------


@dataclass
class Config:
    # --- which clock to trust -------------------------------------------------
    time_col: str = "client timestamp"   # or "record timestamp"

    # --- stroke segmentation --------------------------------------------------
    pause_split_s: float = 0.50          # gap that ends a stroke
    adaptive_pause: bool = True          # override with k * median(dt) if larger
    adaptive_pause_mult: float = 4.0
    split_on_click: bool = True          # a click always terminates a stroke
    max_duration_s: float = 10.0         # hard cap so one stroke can't run away
    max_points: int = 300

    # --- stroke validity ------------------------------------------------------
    min_points: int = 6
    min_path_px: float = 20.0
    min_duration_s: float = 0.05

    # --- feature knobs --------------------------------------------------------
    dir_change_deg: float = 20.0         # angle change that counts as a reversal
    stationary_px: float = 2.0           # segment shorter than this == micro-pause
    tail_frac: float = 0.25              # "final approach" = last 25% of the stroke

    # --- normalisation --------------------------------------------------------
    screen_w: float | None = None        # e.g. 1920 -> positions/lengths in screen units
    screen_h: float | None = None

    # --- misc -----------------------------------------------------------------
    verbose: bool = True


MOVE_STATES = {"move", "drag"}
DOWN_STATES = {"pressed", "down"}
UP_STATES = {"released", "up"}


# ----------------------------------------------------------------------------
# small numeric helpers
# ----------------------------------------------------------------------------


def _safe_div(a, b, fill=0.0):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.full(a.shape, fill, dtype=float)
    m = np.abs(b) > 1e-12
    out[m] = a[m] / b[m]
    return out


def _wrap(a):
    """Wrap angles to (-pi, pi]."""
    return (np.asarray(a) + np.pi) % (2 * np.pi) - np.pi


def _ffill_invalid(values, valid):
    """Replace invalid entries with the last valid one (vectorised)."""
    values = np.asarray(values, dtype=float)
    valid = np.asarray(valid, dtype=bool)
    if not valid.any():
        return np.zeros_like(values)
    idx = np.where(valid, np.arange(len(values)), 0)
    idx = np.maximum.accumulate(idx)
    return values[idx]


def _stats(prefix, arr, pcts=(25, 50, 75)):
    """mean/std/min/max/percentiles of an array, as a flat dict."""
    out = {}
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        keys = ["mean", "std", "min", "max"] + [f"p{p}" for p in pcts]
        return {f"{prefix}_{k}": np.nan for k in keys}
    out[f"{prefix}_mean"] = float(arr.mean())
    out[f"{prefix}_std"] = float(arr.std(ddof=0))
    out[f"{prefix}_min"] = float(arr.min())
    out[f"{prefix}_max"] = float(arr.max())
    for p, q in zip(pcts, np.percentile(arr, pcts)):
        out[f"{prefix}_p{p}"] = float(q)
    return out


# ----------------------------------------------------------------------------
# 1. load
# ----------------------------------------------------------------------------


def load_events(path, cfg: Config = Config()) -> pd.DataFrame:
    """Read a raw log and return a clean frame with columns t, x, y, state, button, is_move."""
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]

    tcol = cfg.time_col.strip().lower()
    if tcol not in df.columns:
        raise KeyError(f"time column {tcol!r} not in {list(df.columns)}")
    for c in ("x", "y", "state"):
        if c not in df.columns:
            raise KeyError(f"expected column {c!r} in {list(df.columns)}")

    out = pd.DataFrame(
        {
            "t": pd.to_numeric(df[tcol], errors="coerce"),
            "x": pd.to_numeric(df["x"], errors="coerce"),
            "y": pd.to_numeric(df["y"], errors="coerce"),
            "state": df["state"].astype(str).str.strip().str.lower(),
            "button": df.get("button", "NoButton").astype(str).str.strip().str.lower(),
        }
    ).dropna(subset=["t", "x", "y"])

    # Balabit sometimes parks the cursor at absurd coordinates on session start/end
    out = out[(out.x.between(-1e4, 1e5)) & (out.y.between(-1e4, 1e5))]

    out = out.sort_values("t", kind="mergesort").reset_index(drop=True)
    out["is_move"] = out.state.isin(MOVE_STATES)

    # Ties: several move samples share one timestamp (event coalescing in the log).
    # Keep the LAST position of each tied group; never drop button events.
    moves = out[out.is_move].drop_duplicates(subset="t", keep="last")
    others = out[~out.is_move]
    out = (
        pd.concat([moves, others])
        .sort_values(["t"], kind="mergesort")
        .reset_index(drop=True)
    )

    if cfg.screen_w:
        out["x"] = out["x"] / cfg.screen_w
    if cfg.screen_h:
        out["y"] = out["y"] / cfg.screen_h

    return out


def diagnose(events: pd.DataFrame, cfg: Config = Config()) -> dict:
    """Sampling-rate report. Run this once per dataset before trusting any derivative."""
    mv = events[events.is_move]
    dt = np.diff(mv.t.values)
    dt = dt[dt > 0]
    d = {
        "n_events": len(events),
        "n_moves": len(mv),
        "n_clicks": int(events.state.isin(DOWN_STATES).sum()),
        "duration_s": float(events.t.max() - events.t.min()) if len(events) else 0.0,
        "dt_median": float(np.median(dt)) if dt.size else np.nan,
        "dt_p90": float(np.percentile(dt, 90)) if dt.size else np.nan,
        "sample_rate_hz": float(1.0 / np.median(dt)) if dt.size else np.nan,
    }
    if cfg.verbose:
        print(
            f"[diagnose] {d['n_moves']} moves, {d['n_clicks']} clicks, "
            f"{d['duration_s']:.0f}s, median dt={d['dt_median']*1000:.0f}ms "
            f"(~{d['sample_rate_hz']:.0f} Hz)"
        )
        if d["sample_rate_hz"] < 30:
            print(
                "  ! low sampling rate: jerk and spectral features will be mostly "
                "quantisation noise. Trust speed/geometry/timing features instead."
            )
        if cfg.pause_split_s < 3 * d["dt_median"]:
            print(
                f"  ! pause_split_s={cfg.pause_split_s}s is close to the sampling "
                f"interval; almost every sample would start a new stroke."
            )
    return d


# ----------------------------------------------------------------------------
# 2. click pairing
# ----------------------------------------------------------------------------


def pair_clicks(events: pd.DataFrame) -> pd.DataFrame:
    """Match each press to the next release of the same button -> dwell time."""
    ev = events[events.state.isin(DOWN_STATES | UP_STATES)]
    rows, open_press = [], {}
    for r in ev.itertuples():
        if r.state in DOWN_STATES:
            open_press[r.button] = r
        elif r.state in UP_STATES and r.button in open_press:
            p = open_press.pop(r.button)
            rows.append(
                {
                    "t_down": p.t,
                    "t_up": r.t,
                    "dwell": r.t - p.t,
                    "button": r.button,
                    "cx": p.x,
                    "cy": p.y,
                }
            )
    return pd.DataFrame(rows, columns=["t_down", "t_up", "dwell", "button", "cx", "cy"])


# ----------------------------------------------------------------------------
# 3. segmentation
# ----------------------------------------------------------------------------


def segment_strokes(events: pd.DataFrame, cfg: Config = Config()):
    """Cut the move stream into strokes. Returns a list of (t, x, y, is_drag, click_row_or_None)."""
    mv = events[events.is_move].reset_index(drop=True)
    if len(mv) < 2:
        return []

    thr = cfg.pause_split_s
    if cfg.adaptive_pause:
        dt_all = np.diff(mv.t.values)
        dt_all = dt_all[dt_all > 0]
        if dt_all.size:
            thr = max(thr, cfg.adaptive_pause_mult * float(np.median(dt_all)))

    t = mv.t.values.astype(float)
    x = mv.x.values.astype(float)
    y = mv.y.values.astype(float)
    is_drag = (mv.state == "drag").values

    # how many click events have happened before each move sample
    click_cum = events.state.isin(DOWN_STATES | UP_STATES).cumsum()
    click_cum_mv = click_cum[events.is_move.values].values

    dt = np.diff(t)
    brk = np.zeros(len(t), dtype=bool)
    brk[1:] |= dt > thr                       # pause
    brk[1:] |= dt <= 0                        # clock glitch
    brk[1:] |= is_drag[1:] != is_drag[:-1]    # drag <-> free move
    if cfg.split_on_click:
        brk[1:] |= np.diff(click_cum_mv) > 0  # a click happened in between

    sid = np.cumsum(brk)

    clicks = pair_clicks(events)
    strokes = []
    for _, idx in pd.Series(np.arange(len(t))).groupby(sid):
        idx = idx.values
        # hard caps: chop over-long strokes into pieces
        pieces = [idx]
        if len(idx) > cfg.max_points:
            pieces = [
                idx[i : i + cfg.max_points] for i in range(0, len(idx), cfg.max_points)
            ]
        for p in pieces:
            if len(p) < 2:
                continue
            tt, xx, yy = t[p], x[p], y[p]
            if tt[-1] - tt[0] > cfg.max_duration_s:
                keep = tt - tt[0] <= cfg.max_duration_s
                tt, xx, yy = tt[keep], xx[keep], yy[keep]
                if len(tt) < 2:
                    continue
            # click that terminates this stroke, if any (press within 1s after the end)
            click = None
            if len(clicks):
                cand = clicks[(clicks.t_down >= tt[-1] - 1e-9) & (clicks.t_down <= tt[-1] + 1.0)]
                if len(cand):
                    click = cand.iloc[0]
            strokes.append((tt, xx, yy, bool(is_drag[p[0]]), click))
    return strokes


# ----------------------------------------------------------------------------
# 4. per-stroke features
# ----------------------------------------------------------------------------


def stroke_features(t, x, y, is_drag=False, click=None, cfg: Config = Config()) -> dict:
    n = len(t)
    f: dict = {}

    dt = np.diff(t)
    dx, dy = np.diff(x), np.diff(y)
    seg = np.hypot(dx, dy)
    v = _safe_div(seg, dt)

    duration = float(t[-1] - t[0])
    path = float(seg.sum())
    disp = float(np.hypot(x[-1] - x[0], y[-1] - y[0]))

    # --- shape ---------------------------------------------------------------
    f["n_points"] = n
    f["duration"] = duration
    f["path_len"] = path
    f["displacement"] = disp
    f["straightness"] = disp / path if path > 0 else 0.0
    f["is_drag"] = int(is_drag)

    bw, bh = float(x.max() - x.min()), float(y.max() - y.min())
    f["bbox_w"], f["bbox_h"] = bw, bh
    f["bbox_area"] = bw * bh
    f["bbox_aspect"] = np.log1p(bw) - np.log1p(bh)          # symmetric, no div-by-zero
    f["angle_start_end"] = float(np.arctan2(y[-1] - y[0], x[-1] - x[0]))
    f["dir_sin"] = float(np.sin(f["angle_start_end"]))      # layout-invariant direction
    f["dir_cos"] = float(np.cos(f["angle_start_end"]))

    # deviation from the straight start->end line ("how bowed is the path")
    if disp > 1e-9:
        ux, uy = (x[-1] - x[0]) / disp, (y[-1] - y[0]) / disp
        perp = np.abs((x - x[0]) * (-uy) + (y - y[0]) * ux)
        f["dev_mean"] = float(perp.mean())
        f["dev_max"] = float(perp.max())
        f["dev_max_norm"] = float(perp.max() / disp)
    else:
        f["dev_mean"] = f["dev_max"] = f["dev_max_norm"] = 0.0

    # --- speed ---------------------------------------------------------------
    f.update(_stats("v", v))
    f["v_cv"] = f["v_std"] / f["v_mean"] if f["v_mean"] else np.nan
    tm = 0.5 * (t[1:] + t[:-1])                              # midpoint times for v
    if v.size:
        f["t_peak_frac"] = float((tm[int(np.argmax(v))] - t[0]) / duration) if duration > 0 else np.nan
        f["v_terminal"] = float(v[-1])
        f["v_initial"] = float(v[0])
    else:
        f["t_peak_frac"] = f["v_terminal"] = f["v_initial"] = np.nan

    # --- acceleration / jerk (weak at low sample rates -- see diagnose()) -----
    if n >= 3:
        a = _safe_div(np.diff(v), np.diff(tm))
        f.update(_stats("a", a))
        f["a_abs_mean"] = float(np.abs(a).mean())
        f["accel_frac"] = float((a > 0).mean())              # share of time speeding up
    else:
        f.update(_stats("a", np.array([])))
        f["a_abs_mean"] = f["accel_frac"] = np.nan

    if n >= 4:
        tj = 0.5 * (tm[1:] + tm[:-1])
        j = _safe_div(np.diff(a), np.diff(tj))
        f["j_abs_mean"] = float(np.abs(j).mean())
        f["j_abs_max"] = float(np.abs(j).max())
        f["j_std"] = float(j.std(ddof=0))
    else:
        f["j_abs_mean"] = f["j_abs_max"] = f["j_std"] = np.nan

    # --- angular ------------------------------------------------------------
    th = _ffill_invalid(np.arctan2(dy, dx), seg > 1e-12)
    if th.size >= 2:
        dth = _wrap(np.diff(th))
        angv = _safe_div(dth, np.diff(tm))
        curv = _safe_div(dth, 0.5 * (seg[1:] + seg[:-1]))
        f["angle_abs_sum"] = float(np.abs(dth).sum())
        f["angle_abs_mean"] = float(np.abs(dth).mean())
        f["angv_abs_mean"] = float(np.abs(angv).mean())
        f["angv_abs_max"] = float(np.abs(angv).max())
        f["curv_abs_mean"] = float(np.abs(curv).mean())
        f["curv_abs_max"] = float(np.abs(curv).max())
        f["curv_std"] = float(curv.std(ddof=0))
        nd = int((np.abs(dth) > np.deg2rad(cfg.dir_change_deg)).sum())
        f["n_dir_changes"] = nd
        f["dir_change_rate"] = nd / duration if duration > 0 else np.nan
    else:
        for k in (
            "angle_abs_sum angle_abs_mean angv_abs_mean angv_abs_max "
            "curv_abs_mean curv_abs_max curv_std n_dir_changes dir_change_rate"
        ).split():
            f[k] = np.nan

    # --- sampling / rhythm ---------------------------------------------------
    f["dt_mean"] = float(dt.mean())
    f["dt_std"] = float(dt.std(ddof=0))
    f["dt_cv"] = f["dt_std"] / f["dt_mean"] if f["dt_mean"] else np.nan
    micro = seg < cfg.stationary_px
    f["n_micro_pauses"] = int(micro.sum())
    f["micro_pause_frac"] = float(dt[micro].sum() / duration) if duration > 0 else np.nan

    # --- final approach (the part that discriminates most, per the literature) -
    tail = tm >= (t[-1] - cfg.tail_frac * duration) if duration > 0 else np.zeros_like(tm, bool)
    if tail.any():
        f["tail_v_mean"] = float(v[tail].mean())
        f["tail_v_max"] = float(v[tail].max())
        f["tail_path_frac"] = float(seg[tail].sum() / path) if path > 0 else np.nan
    else:
        f["tail_v_mean"] = f["tail_v_max"] = f["tail_path_frac"] = np.nan

    # --- click ---------------------------------------------------------------
    if click is not None:
        cx, cy = float(click.cx), float(click.cy)
        d = np.hypot(x - cx, y - cy)
        i = int(np.argmin(d))
        f["has_click"] = 1
        f["click_dwell"] = float(click.dwell)
        f["click_button_left"] = int(str(click.button).startswith("left"))
        f["click_delay"] = float(click.t_down - t[-1])       # move end -> press
        f["click_dist_end"] = float(np.hypot(x[-1] - cx, y[-1] - cy))
        f["overshoot_path"] = float(seg[i:].sum())           # path travelled after closest approach
        f["overshoot_max"] = float(d[i:].max())
    else:
        for k in (
            "has_click click_dwell click_button_left click_delay "
            "click_dist_end overshoot_path overshoot_max"
        ).split():
            f[k] = 0 if k == "has_click" else np.nan

    return f


# ----------------------------------------------------------------------------
# 5. file / directory drivers
# ----------------------------------------------------------------------------


def strokes_to_frame(strokes, cfg: Config = Config()) -> pd.DataFrame:
    rows, dropped = [], {"short": 0, "tiny_path": 0, "brief": 0}
    for k, (t, x, y, is_drag, click) in enumerate(strokes):
        if len(t) < cfg.min_points:
            dropped["short"] += 1
            continue
        if float(np.hypot(np.diff(x), np.diff(y)).sum()) < cfg.min_path_px:
            dropped["tiny_path"] += 1
            continue
        if t[-1] - t[0] < cfg.min_duration_s:
            dropped["brief"] += 1
            continue
        f = stroke_features(t, x, y, is_drag, click, cfg)
        f["stroke_id"] = k
        f["t_start"] = float(t[0])
        f["t_end"] = float(t[-1])
        rows.append(f)

    if cfg.verbose:
        print(
            f"[strokes] kept {len(rows)} / {len(strokes)} "
            f"(dropped: {dropped['short']} too few points, "
            f"{dropped['tiny_path']} too short, {dropped['brief']} too brief)"
        )
    df = pd.DataFrame(rows)
    return df.replace([np.inf, -np.inf], np.nan)


def process_file(path, cfg: Config = Config(), user=None, session=None) -> pd.DataFrame:
    ev = load_events(path, cfg)
    if cfg.verbose:
        diagnose(ev, cfg)
    df = strokes_to_frame(segment_strokes(ev, cfg), cfg)
    if len(df):
        df.insert(0, "session", session or Path(path).name)
        df.insert(0, "user", user or Path(path).parent.name)
    return df


def process_dir(root, cfg: Config = Config(), pattern="**/session_*") -> pd.DataFrame:
    """Walk a Balabit-style tree (root/userN/session_xxxx) into one long frame."""
    root = Path(root)
    out = []
    for p in sorted(root.glob(pattern)):
        if not p.is_file():
            continue
        try:
            d = process_file(p, cfg, user=p.parent.name, session=p.name)
            if len(d):
                out.append(d)
        except Exception as e:  # keep going -- some session files are truncated
            warnings.warn(f"{p}: {e}")
    if not out:
        return pd.DataFrame()
    df = pd.concat(out, ignore_index=True)
    if cfg.verbose:
        print(f"[process_dir] {len(df)} strokes from {df.user.nunique()} users")
    return df


# ----------------------------------------------------------------------------
# 6. optional: strokes -> model input windows
# ----------------------------------------------------------------------------


NON_FEATURE = {"user", "session", "stroke_id", "t_start", "t_end"}


def strokes_to_windows(
    df: pd.DataFrame,
    n: int = 25,
    stride: int = 5,
    min_strokes: int = 8,
    max_span_s: float = 60.0,
    aggs=("mean", "std", "p25", "p50", "p75"),
) -> pd.DataFrame:
    """Sliding window over strokes -> one wide vector per window.

    Windows spanning more than max_span_s are dropped (the user walked away).
    Windows with fewer than min_strokes are never emitted (idle reading).
    """
    feat_cols = [c for c in df.columns if c not in NON_FEATURE]
    rows = []
    for (u, s), g in df.groupby(["user", "session"], sort=False):
        g = g.sort_values("t_start").reset_index(drop=True)
        vals = g[feat_cols].to_numpy(dtype=float)
        for end in range(n, len(g) + 1, stride):
            start = end - n
            if end - start < min_strokes:
                continue
            span = g.t_end.iloc[end - 1] - g.t_start.iloc[start]
            if span > max_span_s:
                continue
            blk = vals[start:end]
            rec = {"user": u, "session": s,
                   "t_start": float(g.t_start.iloc[start]),
                   "t_end": float(g.t_end.iloc[end - 1]),
                   "n_strokes": end - start, "span_s": float(span)}
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                for agg in aggs:
                    if agg == "mean":
                        vv = np.nanmean(blk, axis=0)
                    elif agg == "std":
                        vv = np.nanstd(blk, axis=0)
                    else:
                        vv = np.nanpercentile(blk, int(agg[1:]), axis=0)
                    for c, val in zip(feat_cols, vv):
                        rec[f"{c}_{agg}"] = float(val)
            rows.append(rec)
    out = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)
    print(f"[windows] {len(out)} windows x {len([c for c in out.columns if c not in NON_FEATURE | {'n_strokes','span_s'}])} features")
    return out


# ----------------------------------------------------------------------------


import sys

if __name__ == "__main__" and "ipykernel" not in sys.modules:

    cfg = Config()
    src = Path(sys.argv[1]) if len(sys.argv) > 1 else None
    if src is None:
        print(__doc__)
    elif src.is_dir():
        d = process_dir(src, cfg)
        d.to_csv("strokes.csv", index=False)
    else:
        d = process_file(src, cfg)
        d.to_csv("strokes.csv", index=False)
        print(d.head())

---
## 1. Event-level cleaning

`load_events()` in the cell above already did the bare minimum (sort, drop ties,
clip absurd coordinates). This replaces it with the full clean, and makes the same
cleaning reachable from an **in-memory** DataFrame, which is what the ./data loader
gives us.

What gets removed and why:

| step | what it kills | why it matters |
|---|---|---|
| `dropna` | rows with unparseable `t/x/y` | one NaN poisons `np.diff` for the whole stroke |
| `drop_scroll` | wheel events | they carry `Down`/`Up` states and would be paired as fake clicks with fake dwell times |
| range clip | cursor parked at `(-1e6, …)` | Balabit does this at session start/end |
| out-of-order | rows where the clock steps backwards | sorting them into place is worse than dropping — it reorders real motion |
| tied timestamps | coalesced move samples | `dt = 0` → infinite velocity |
| despike | isolated 1-sample position jumps | remote-desktop capture artifact: out and immediately back. Detected as `d(i-1,i)` and `d(i,i+1)` both large while `d(i-1,i+1)` is small |
| `collapse_frozen` | repeated identical positions | **off by default** — a frozen cursor is real behaviour (the user stopped), and `micro_pause_frac` is built to measure it |

Not done on purpose: **no smoothing / Savitzky-Golay filter.** Any low-pass filter
destroys exactly the high-frequency content that `j_abs_mean`, `j_std` and `dt_cv`
are trying to measure. At ~60 Hz those features are already marginal; filtering
would make them look clean and mean nothing.

In [ ]:
from dataclasses import dataclass, asdict, replace
import numpy as np, pandas as pd, warnings, json, time

@dataclass
class PrepConfig:
    # --- event level ---
    x_range: tuple = (-2000.0, 8000.0)
    y_range: tuple = (-2000.0, 5000.0)
    drop_scroll: bool = True
    drop_out_of_order: bool = True
    despike: bool = True
    despike_px: float = 250.0
    despike_ratio: float = 0.35
    collapse_frozen: bool = False
    frozen_px: float = 0.0
    # --- session level ---
    min_events: int = 150
    min_moves: int = 100
    min_duration_s: float = 20.0
    max_dt_median_s: float = 0.40
    # --- stroke level ---
    max_speed_px_s: float = 6000.0
    max_path_px: float = 20000.0
    drop_zero_var_strokes: bool = True
    # --- feature level ---
    max_nan_frac: float = 0.50
    min_unique: int = 3
    winsor_mad: float = 6.0
    log1p_skew: float = 3.0
    scaler: str = "robust"          # "robust" | "standard" | "none"
    corr_prune: float = 0.0         # 0 disables; else drop |r| above this


def _despike_mask(x, y, jump_px, ratio):
    n = len(x)
    bad = np.zeros(n, dtype=bool)
    if n < 3:
        return bad
    d = np.hypot(np.diff(x), np.diff(y))
    a, b = d[:-1], d[1:]
    c = np.hypot(x[2:] - x[:-2], y[2:] - y[:-2])
    bad[1:-1] = (a > jump_px) & (b > jump_px) & (c < ratio * (a + b))
    return bad


def clean_events(raw: pd.DataFrame, cfg: Config = Config(), pcfg: PrepConfig = PrepConfig()):
    rep = {"n_raw": len(raw)}
    df = raw.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    tcol = cfg.time_col.strip().lower()
    for c in (tcol, "x", "y", "state"):
        if c not in df.columns:
            raise KeyError(f"expected column {c!r} in {list(df.columns)}")

    out = pd.DataFrame({
        "t":      pd.to_numeric(df[tcol], errors="coerce").astype("float64"),
        "x":      pd.to_numeric(df["x"], errors="coerce").astype("float64"),
        "y":      pd.to_numeric(df["y"], errors="coerce").astype("float64"),
        "state":  df["state"].astype(str).str.strip().str.lower(),
        "button": (df["button"] if "button" in df.columns else "nobutton").astype(str).str.strip().str.lower(),
    })

    n = len(out); out = out.dropna(subset=["t", "x", "y"]);            rep["drop_nan"] = n - len(out)

    if pcfg.drop_scroll:
        n = len(out)
        out = out[~(out.button.str.contains("scroll") | out.state.str.contains("scroll"))]
        rep["drop_scroll"] = n - len(out)

    n = len(out)
    out = out[out.x.between(*pcfg.x_range) & out.y.between(*pcfg.y_range)];  rep["drop_oob"] = n - len(out)

    if pcfg.drop_out_of_order and len(out):
        t = out.t.values
        keep = t >= np.maximum.accumulate(t) - 1e-9
        rep["drop_backwards"] = int((~keep).sum())
        out = out[keep]
    else:
        rep["drop_backwards"] = 0

    out = out.sort_values("t", kind="mergesort").reset_index(drop=True)
    out["is_move"] = out.state.isin(MOVE_STATES)

    n = len(out)
    moves  = out[out.is_move].drop_duplicates(subset="t", keep="last")
    others = out[~out.is_move]
    out = pd.concat([moves, others]).sort_values("t", kind="mergesort").reset_index(drop=True)
    rep["drop_tied_t"] = n - len(out)

    if pcfg.despike and out.is_move.any():
        mv = out.index[out.is_move.values].to_numpy()
        bad = _despike_mask(out.x.values[mv], out.y.values[mv], pcfg.despike_px, pcfg.despike_ratio)
        rep["drop_spikes"] = int(bad.sum())
        if bad.any():
            out = out.drop(index=mv[bad]).reset_index(drop=True)
    else:
        rep["drop_spikes"] = 0

    if pcfg.collapse_frozen and out.is_move.any():
        mv = out.is_move.values
        same = np.zeros(len(out), dtype=bool)
        same[1:] = mv[1:] & mv[:-1] & (np.abs(np.diff(out.x.values)) <= pcfg.frozen_px) \
                                    & (np.abs(np.diff(out.y.values)) <= pcfg.frozen_px)
        rep["drop_frozen"] = int(same.sum())
        out = out[~same].reset_index(drop=True)
    else:
        rep["drop_frozen"] = 0

    if cfg.screen_w: out["x"] = out["x"] / cfg.screen_w
    if cfg.screen_h: out["y"] = out["y"] / cfg.screen_h

    rep["n_clean"] = len(out)
    return out.reset_index(drop=True), rep


def session_report(ev: pd.DataFrame, pcfg: PrepConfig = PrepConfig()) -> dict:
    mv = ev[ev.is_move]
    dt = np.diff(mv.t.values); dt = dt[dt > 0]
    d = {
        "n_events":  len(ev),
        "n_moves":   len(mv),
        "n_clicks":  int(ev.state.isin(DOWN_STATES).sum()),
        "duration_s": float(ev.t.max() - ev.t.min()) if len(ev) else 0.0,
        "dt_median": float(np.median(dt)) if dt.size else np.nan,
        "hz":        float(1 / np.median(dt)) if dt.size else np.nan,
    }
    reasons = []
    if d["n_events"]   < pcfg.min_events:      reasons.append("few_events")
    if d["n_moves"]    < pcfg.min_moves:       reasons.append("few_moves")
    if d["duration_s"] < pcfg.min_duration_s:  reasons.append("short")
    if not np.isfinite(d["dt_median"]) or d["dt_median"] > pcfg.max_dt_median_s:
        reasons.append("slow_sampling")
    d["keep"] = len(reasons) == 0
    d["drop_reason"] = ",".join(reasons)
    return d

---
## 2. Connect the loader to the feature extractor

The ./data loader produced `users['user7']['session_123'] -> DataFrame`, but
`process_file()` takes a *path*. `process_frame()` is the missing link, and
`load_events` is rebound so the path-based entry points (`process_file`,
`process_dir`) clean identically — one code path, no drift.

Runs `clean_events → session QC → segment_strokes → stroke_features` over
everything loaded, and returns a per-session QC table alongside the stroke table
so dropped sessions are auditable rather than silent.

In [ ]:
PREP = PrepConfig()

def load_events(path, cfg: Config = Config()) -> pd.DataFrame:      # noqa: F811
    """Overrides the version in the feature cell so file- and memory-paths clean identically."""
    return clean_events(pd.read_csv(path), cfg, PREP)[0]


def process_frame(raw: pd.DataFrame, user: str, session: str,
                  cfg: Config = Config(), pcfg: PrepConfig = PREP):
    ev, rep = clean_events(raw, cfg, pcfg)
    qc = {"user": user, "session": session, **rep, **session_report(ev, pcfg)}
    if not qc["keep"]:
        return pd.DataFrame(), qc
    st = strokes_to_frame(segment_strokes(ev, cfg), cfg)
    qc["n_strokes"] = len(st)
    if len(st):
        st.insert(0, "session", session)
        st.insert(0, "user", user)
    else:
        qc["keep"], qc["drop_reason"] = False, "no_strokes"
    return st, qc


def process_loaded(users: dict, cfg: Config = Config(), pcfg: PrepConfig = PREP,
                   progress_every: int = 100):
    cfg = replace(cfg, verbose=False)
    frames, qcs, t0, i = [], [], time.time(), 0
    total = sum(len(s) for s in users.values())
    for u, sessions in users.items():
        for sname, raw in sessions.items():
            i += 1
            try:
                st, qc = process_frame(raw, u, sname, cfg, pcfg)
            except Exception as e:
                qcs.append({"user": u, "session": sname, "keep": False,
                            "drop_reason": f"error:{type(e).__name__}"})
                warnings.warn(f"{u}/{sname}: {e}")
                continue
            qcs.append(qc)
            if len(st):
                frames.append(st)
            if progress_every and i % progress_every == 0:
                print(f"  {i}/{total} sessions  ({time.time()-t0:.0f}s)")
    qc = pd.DataFrame(qcs)
    strokes = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print(f"[pipeline] {qc.keep.sum()}/{len(qc)} sessions kept, "
          f"{len(strokes):,} strokes, {time.time()-t0:.0f}s")
    if (~qc.keep).any():
        print(qc.loc[~qc.keep, "drop_reason"].value_counts().to_string())
    return strokes, qc


# --- run it -----------------------------------------------------------------
cfg  = Config(time_col="client timestamp", verbose=False)
PREP = PrepConfig()

strokes_raw, qc = process_loaded(users, cfg, PREP, progress_every=100)
display(qc.head())
display(strokes_raw.head())

---
## 3. Stroke-table QC

Session-level QC kept or killed whole files; this drops individual bad strokes
that survived segmentation. `max_speed_px_s = 6000` is the operative one — real
sustained cursor motion tops out around 3000–4000 px/s, so anything above that
is a residual teleport the despiker missed.

**The tradeoff worth being explicit about:** this is a biometrics dataset, and an
"outlier" here is not necessarily noise — a user who flings the mouse is *supposed*
to look extreme. Every threshold below trades away some real between-user variance
to remove capture artifacts. Keep them loose (kill physics violations, not unusual
humans) and check the per-user drop rate: if one user loses far more strokes than
the rest, the filter is eating signal, not noise.

In [ ]:
def filter_strokes(df: pd.DataFrame, pcfg: PrepConfig = PREP) -> pd.DataFrame:
    n0 = len(df); drops = {}
    m = pd.Series(True, index=df.index)

    bad = df.v_max > pcfg.max_speed_px_s;        drops["impossible_speed"] = int(bad.sum()); m &= ~bad
    bad = df.path_len > pcfg.max_path_px;        drops["huge_path"] = int(bad.sum());        m &= ~bad
    if pcfg.drop_zero_var_strokes:
        bad = (df.v_std == 0) | (df.bbox_area == 0)
        drops["degenerate"] = int(bad.sum()); m &= ~bad
    bad = ~np.isfinite(df.duration) | (df.duration <= 0)
    drops["bad_duration"] = int(bad.sum()); m &= ~bad

    out = df[m].reset_index(drop=True)
    print(f"[strokes] {len(out):,}/{n0:,} kept  " +
          "  ".join(f"-{k}:{v}" for k, v in drops.items() if v))
    return out


strokes = filter_strokes(strokes_raw, PREP)

# per-user drop rate -- should be roughly flat across users
before = strokes_raw.groupby("user").size()
after  = strokes.groupby("user").size()
display(pd.DataFrame({"raw": before, "kept": after,
                      "drop_%": (100 * (1 - after / before)).round(1)}))

---
## 4. Session-level train/test split

Do this **before** fitting any scaler or imputer.

Two leaks to avoid, both of which will hand you 99% accuracy that collapses on real
data:

1. **Row-level splitting.** Consecutive strokes in one session are nearly
   duplicates. A random row split puts near-copies on both sides, so the model
   memorises sessions rather than users. Split by session.
2. **Fitting the preprocessor on everything.** Medians, MAD bounds and scale
   factors computed over train+test leak test distribution into training.
   Fit on train, transform test.

Balabit's own protocol goes further: train on a user's sessions, test on *held-out
sessions*, and for the impostor task test on sessions from users never seen in
training. If that's your task, split by user, not by session.

In [ ]:
def session_split(df: pd.DataFrame, test_frac: float = 0.3, seed: int = 0):
    rng = np.random.default_rng(seed)
    tr, te, notes = [], [], []
    for u, g in df.groupby("user", sort=True):
        sess = np.array(sorted(g.session.unique()))
        if len(sess) < 2:
            # only one session for this user: fall back to a chronological split
            gg = g.sort_values("t_start")
            cut = int(len(gg) * (1 - test_frac))
            tr.append(gg.iloc[:cut]); te.append(gg.iloc[cut:])
            notes.append(f"{u}: 1 session -> time split (within-session leakage possible)")
            continue
        rng.shuffle(sess)
        k = max(1, int(round(len(sess) * test_frac)))
        te_s = set(sess[:k])
        tr.append(g[~g.session.isin(te_s)]); te.append(g[g.session.isin(te_s)])
    train = pd.concat(tr, ignore_index=True)
    test  = pd.concat(te, ignore_index=True)
    for n in notes: print("  !", n)
    print(f"[split] train {len(train):,} strokes / {train.session.nunique()} sessions | "
          f"test {len(test):,} / {test.session.nunique()} sessions | "
          f"users {train.user.nunique()}")
    assert set(train.session) & set(test.session) == set() or notes, "session leak"
    return train, test


train_s, test_s = session_split(strokes, test_frac=0.30, seed=0)

**félkövér szöveg**---
## 5. Fit preprocessing on train, apply to both

Order is: drop dead columns → `log1p` heavy tails → winsorize → impute → scale.

- **Winsorize, don't drop.** Clipping at median ± 6·MAD keeps the row and its label;
  dropping rows throws away strokes and biases the class balance toward whoever
  happens to be tidy. MAD is used rather than std because std is itself dragged
  around by the outliers it is meant to detect.
- **`log1p` before clipping.** `path_len`, `j_abs_max`, `curv_abs_max` etc. are
  log-normal-ish; on the raw scale a 6·MAD bound clips the top decile of a
  perfectly ordinary distribution.
- **Structural NaNs are not missing data.** `click_dwell` is NaN for strokes with
  no click — roughly a third of them. Median-imputing fabricates a dwell time for
  a click that never happened. Those columns fill with 0 and `has_click` carries
  the information; everything else fills with the train median.
- **`scaler="robust"`** (median / IQR) by default. Irrelevant for trees, essential
  for SVM/kNN/NN.
- **`corr_prune`** is off by default. It helps linear models and hurts nothing for
  trees, but it silently decides which of two correlated features to keep.

In [ ]:
NON_FEATURE_ALL = {"user", "session", "stroke_id", "t_start", "t_end", "n_strokes", "span_s"}
# structurally-missing columns: NaN means "no click happened", not "unknown"
STRUCTURAL_NAN = {"click_dwell", "click_button_left", "click_delay", "click_dist_end",
                  "overshoot_path", "overshoot_max"}


def fit_preprocessor(train: pd.DataFrame, pcfg: PrepConfig = PREP) -> dict:
    feats = [c for c in train.columns if c not in NON_FEATURE_ALL]
    X = train[feats].astype("float64")

    keep = [c for c in feats
            if X[c].isna().mean() <= pcfg.max_nan_frac
            and X[c].nunique(dropna=True) >= pcfg.min_unique]
    dropped_cols = sorted(set(feats) - set(keep))
    X = X[keep]

    # heavy right tails -> log1p (only strictly non-negative columns)
    logs = [c for c in keep
            if (X[c].dropna() >= 0).all() and abs(X[c].skew(skipna=True)) > pcfg.log1p_skew]
    X[logs] = np.log1p(X[logs])

    # robust bounds from median / MAD, percentile fallback when MAD collapses
    med = X.median()
    mad = (X - med).abs().median() * 1.4826
    lo = med - pcfg.winsor_mad * mad
    hi = med + pcfg.winsor_mad * mad
    flat = mad <= 1e-12
    if flat.any():
        lo[flat] = X.loc[:, flat].quantile(0.005)
        hi[flat] = X.loc[:, flat].quantile(0.995)
    X = X.clip(lo, hi, axis=1)

    fill = {c: (0.0 if c in STRUCTURAL_NAN else float(med[c])) for c in keep}

    if pcfg.scaler == "robust":
        center = X.median()
        q1, q3 = X.quantile(0.25), X.quantile(0.75)
        scale = (q3 - q1).replace(0, np.nan).fillna(X.std(ddof=0)).replace(0, 1.0)
    elif pcfg.scaler == "standard":
        center, scale = X.mean(), X.std(ddof=0).replace(0, 1.0)
    else:
        center = pd.Series(0.0, index=keep); scale = pd.Series(1.0, index=keep)

    pre = {"features": keep, "dropped_cols": dropped_cols, "log_cols": logs,
           "lo": lo.to_dict(), "hi": hi.to_dict(), "fill": fill,
           "center": center.to_dict(), "scale": scale.to_dict(),
           "config": asdict(pcfg)}

    if pcfg.corr_prune:
        Z = apply_preprocessor(train, pre)
        cm = Z.corr().abs().to_numpy(copy=True)
        np.fill_diagonal(cm, 0.0)
        drop, cols = set(), list(Z.columns)
        for i in range(len(cols)):
            if cols[i] in drop: continue
            for j in range(i + 1, len(cols)):
                if cm[i, j] > pcfg.corr_prune: drop.add(cols[j])
        pre["features"] = [c for c in keep if c not in drop]
        pre["corr_pruned"] = sorted(drop)
        print(f"[prep] corr>|{pcfg.corr_prune}| pruned {len(drop)} columns")

    print(f"[prep] {len(pre['features'])} features "
          f"(dropped {len(dropped_cols)} constant/empty, log1p on {len(logs)})")
    return pre


def apply_preprocessor(df: pd.DataFrame, pre: dict) -> pd.DataFrame:
    """Returns FEATURES ONLY, row order preserved -- no user/session/t_start columns.

    Keeping metadata out of the matrix is deliberate: `t_start` is a position in the
    session, and a model handed that column will happily learn it.
    """
    cols = pre["features"]
    X = df.reindex(columns=cols).astype("float64")
    logc = [c for c in pre["log_cols"] if c in cols]
    if logc:
        X[logc] = np.log1p(X[logc].clip(lower=-0.999999))
    X = X.clip(pd.Series(pre["lo"])[cols], pd.Series(pre["hi"])[cols], axis=1)
    X = X.fillna(pd.Series(pre["fill"])[cols])
    X = (X - pd.Series(pre["center"])[cols]) / pd.Series(pre["scale"])[cols]
    return X.replace([np.inf, -np.inf], 0.0).reset_index(drop=True)


pre     = fit_preprocessor(train_s, PREP)
X_train = apply_preprocessor(train_s, pre)      # features only
X_test  = apply_preprocessor(test_s,  pre)

y_train, y_test = train_s["user"].to_numpy(), test_s["user"].to_numpy()
g_train, g_test = train_s["session"].to_numpy(), test_s["session"].to_numpy()   # for GroupKFold
print(X_train.shape, X_test.shape)
display(X_train.describe().loc[["mean", "std", "min", "max"]].T.head(10))

---
## 6. Optional: stroke windows

One stroke is a weak signal; most of the Balabit literature aggregates. Build
windows from the **unscaled** stroke table, then split and scale the windows the
same way (aggregating already-scaled features would mean the window stats were
computed on a distribution fitted to individual strokes).

`max_span_s` guards against a window of 25 strokes that spans ten minutes because
the user walked away.

One quirk in `strokes_to_windows` as written: windows are always exactly `n`
strokes (`start = end - n`), so the `min_strokes` argument never fires and the
trailing partial window of each session is dropped. Harmless, but don't expect
`min_strokes=8` to do anything.

In [ ]:
windows = strokes_to_windows(strokes, n=25, stride=5, min_strokes=8, max_span_s=60.0)

if len(windows):
    train_w, test_w = session_split(windows, test_frac=0.30, seed=0)
    pre_w    = fit_preprocessor(train_w, PREP)
    Xw_train = apply_preprocessor(train_w, pre_w)
    Xw_test  = apply_preprocessor(test_w,  pre_w)
    yw_train, yw_test = train_w["user"].to_numpy(), test_w["user"].to_numpy()
    print(Xw_train.shape, Xw_test.shape)

---
## 7. Save to ./data

Parquet, not CSV: preserves dtypes, ~5–10× smaller, and loads in seconds instead of
re-walking 1700 CSVs off disk.

The `preprocessor.joblib` is the important artifact — it holds the clip bounds, fill
values and scale factors **fitted on train**. Any future session must go through the
same object, or inference silently sees a different feature space than training did.
The manifest records both configs so a result can be traced back to the settings that
produced it; bump `tag` when you change a threshold instead of overwriting.

In [ ]:
import joblib

def save_dataset(outdir, *, strokes=None, windows=None, train=None, test=None,
                 pre=None, qc=None, cfg=None, pcfg=None, tag="v1"):
    out = Path(outdir) / tag
    out.mkdir(parents=True, exist_ok=True)
    written = {}

    def _pq(name, df):
        if df is None or not len(df): return
        p = out / f"{name}.parquet"
        d = df.copy()
        for c in ("user", "session"):
            if c in d.columns: d[c] = d[c].astype(str)
        d.to_parquet(p, index=False, compression="zstd")
        written[name] = (str(p), d.shape, p.stat().st_size / 1e6)

    _pq("strokes", strokes); _pq("windows", windows)
    _pq("train", train);     _pq("test", test); _pq("session_qc", qc)

    if pre is not None:
        joblib.dump(pre, out / "preprocessor.joblib")
        (out / "preprocessor.json").write_text(json.dumps(pre, indent=2, default=float))
        written["preprocessor"] = (str(out / "preprocessor.joblib"), None, None)

    manifest = {
        "tag": tag,
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "config": asdict(cfg) if cfg is not None else None,
        "prep_config": asdict(pcfg) if pcfg is not None else None,
        "shapes": {k: v[1] for k, v in written.items() if v[1]},
        "n_features": len(pre["features"]) if pre else None,
    }
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))

    print(f"[save] -> {out}")
    for k, (p, shape, mb) in written.items():
        print(f"   {k:12s} {str(shape):16s} {f'{mb:.1f} MB' if mb else ''}")
    return out


def load_dataset(outdir, tag="v1"):
    out = Path(outdir) / tag
    got = {p.stem: pd.read_parquet(p) for p in out.glob("*.parquet")}
    pj = out / "preprocessor.joblib"
    if pj.exists(): got["pre"] = joblib.load(pj)
    print(f"[load] {out}: " + ", ".join(f"{k}{getattr(v,'shape','')}" for k, v in got.items()))
    return got


SAVE_ROOT = ROOT / "processed"          # -> ./data/processed

save_dataset(
    SAVE_ROOT, tag="v1",
    strokes=strokes,
    windows=windows if "windows" in dir() and len(windows) else None,
    train=X_train.assign(user=y_train, session=g_train),
    test=X_test.assign(user=y_test, session=g_test),
    pre=pre, qc=qc, cfg=cfg, pcfg=PREP,
)

# later sessions start here instead of re-reading 1700 CSVs:
# d   = load_dataset(SAVE_ROOT, "v1")
# pre = d["pre"]; train = d["train"]; test = d["test"]

---
# 8. Siamese embedding model

Goal: a function `window of mouse activity -> vector`, trained so that
`cosine(z_a, z_b)` is high when two windows come from the same person and low
otherwise. Once that holds, everything else is arithmetic on vectors — enrol a
user by averaging their embeddings, verify a session by comparing against that
average, flag an intruder when similarity drops below a threshold.

**What is actually being embedded.** Not a user, and not a single stroke. A
*window* of 25 consecutive strokes from one session. One stroke is far too weak —
one mouse movement from two different people looks nearly identical. The *user*
embedding is a derived object: the mean of that user's window embeddings over
their enrolment sessions (`templates()` below). Keeping the model at window level
means new users can be enrolled without retraining, which is the whole point of
the metric-learning setup over a plain 10-way classifier.

**Architecture choices, and why:**

| choice | what I did | why |
|---|---|---|
| input | the 60-dim preprocessed stroke vectors, `(25, 60)` per window | reuses the whole pipeline above. See the caveat below about sequence input |
| encoder | per-stroke MLP → pool over the stroke axis → 64-d, L2-normalised | |
| pooling | attention + mean + std, concatenated | **permutation invariant on purpose.** Stroke order within a window is close to arbitrary, and a GRU/transformer over it mostly learns session-specific sequencing, which is exactly what fails to transfer to a new session. The `std` branch matters — *consistency* is as identifying as average speed |
| loss | batch-hard triplet with online mining | the modern form of the siamese idea. Random pairs are ~95% trivially-easy after a few epochs and the gradient dies. `loss="contrastive"` gives the classic pair loss if you want the comparison |
| batching | P users × K windows, K spread across distinct sessions | without this a random batch rarely contains a positive pair and there is nothing to mine. Spreading K over sessions makes the hardest positive a *cross-session* pair, which is the hard case that matters |
| output | L2-normalised, cosine similarity | fixes the scale so one threshold works across users |

**The main thing you may want to overrule.** The encoder eats the hand-crafted
feature vectors, not the raw trajectories. That reuses everything above and trains
in a couple of minutes on CPU, but it caps the model at whatever `stroke_features()`
already measured — the network cannot discover a cue the feature code did not encode.
The stronger version resamples each stroke to a fixed-length `(T, 4)` sequence of
`(dx, dy, dt, v)` and puts a 1D CNN under the same pooling head. That needs new
plumbing back to the event level, since the current pipeline discards raw points
after feature extraction. Say the word and I'll add it — it is maybe 40 lines and a
`raw_strokes` return from `segment_strokes`.

In [ ]:
# Colab already has torch; uncomment if the runtime is bare.
# !pip -q install torch

import json, math, time, warnings
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NON_FEATURE_ALL |= {"fold"}      # so the fold tag never becomes a feature
print("device:", DEVICE)


@dataclass
class ModelConfig:
    # --- windowing ---
    n_strokes: int = 25            # strokes per window (the unit that gets embedded)
    stride: int = 8
    max_span_s: float = 90.0       # drop windows where the user walked away mid-way
    # --- architecture ---
    d_hidden: int = 256
    d_embed: int = 64
    n_layers: int = 3
    dropout: float = 0.15
    pool: str = "attn+meanstd"     # "mean" | "meanstd" | "attn+meanstd"
    # --- loss ---
    loss: str = "batch_hard"       # "batch_hard" | "contrastive"
    margin: float = 0.25           # 0 -> soft-margin (log1p(exp(dp-dn))), no tuning
    # --- optimisation ---
    P_users: int = 8               # users per batch
    K_windows: int = 6             # windows per user per batch  (batch = P*K)
    lr: float = 2e-3
    weight_decay: float = 1e-2
    epochs: int = 60
    steps_per_epoch: int = 60
    patience: int = 12
    seed: int = 0


def assign_folds(strokes: pd.DataFrame, val_frac=0.15, test_frac=0.20,
                 seed=0, holdout_users=()) -> pd.DataFrame:
    """Tag every SESSION with train/val/test (or 'unseen' for held-out users).

    Done at session level and before the preprocessor is fitted, so the fold a
    window belongs to is fixed by its session and nothing leaks across folds.
    """
    rng = np.random.default_rng(seed)
    out = strokes.copy()
    fold = pd.Series("train", index=out.index, dtype=object)
    for u, g in out.groupby("user", sort=True):
        if u in holdout_users:
            fold[g.index] = "unseen"; continue
        us = np.array(sorted(g.session.unique())); rng.shuffle(us)
        n_te = max(1, int(round(len(us) * test_frac)))
        n_va = max(1, int(round(len(us) * val_frac))) if len(us) - n_te >= 2 else 0
        tag = {s: "test" for s in us[:n_te]}
        tag |= {s: "val" for s in us[n_te:n_te + n_va]}
        tag |= {s: "train" for s in us[n_te + n_va:]}
        fold[g.index] = g.session.map(tag).values
    out["fold"] = fold
    print(out.groupby("fold").agg(strokes=("user", "size"), sessions=("session", "nunique"),
                                  users=("user", "nunique")).to_string())
    return out


def build_windows(strokes: pd.DataFrame, pre: dict, mcfg: ModelConfig):
    """(M, n_strokes, D) float32 stack of preprocessed stroke vectors + labels/groups/folds."""
    X, y, sess, spans, folds = [], [], [], [], []
    for (u, s), g in strokes.groupby(["user", "session"], sort=False):
        g = g.sort_values("t_start")
        V = apply_preprocessor(g, pre).to_numpy(dtype=np.float32)
        t0, t1 = g.t_start.to_numpy(), g.t_end.to_numpy()
        for end in range(mcfg.n_strokes, len(g) + 1, mcfg.stride):
            st = end - mcfg.n_strokes
            span = t1[end - 1] - t0[st]
            if span > mcfg.max_span_s:
                continue
            X.append(V[st:end]); y.append(u); sess.append(s); spans.append(span)
            folds.append(g["fold"].iloc[0] if "fold" in g.columns else "train")
    if not X:
        raise RuntimeError("no windows -- lower n_strokes or raise max_span_s")
    X = np.stack(X)
    y = np.array(y); sess = np.array(sess); folds = np.array(folds)
    print(f"[windows] {X.shape[0]:,} windows x {X.shape[1]} strokes x {X.shape[2]} feats "
          f"| {len(np.unique(y))} users | median span {np.median(spans):.1f}s")
    cnt = pd.Series(y).value_counts()
    print(f"[windows] per-user: min {cnt.min()}, median {int(cnt.median())}, max {cnt.max()}")
    return X, y, sess, folds


def fold_index(folds, sess, y):
    out = {f: np.where(folds == f)[0] for f in ("train", "val", "test", "unseen")
           if (folds == f).any()}
    print("[split] " + " | ".join(
        f"{k}: {len(v):,} win / {len(set(sess[v]))} sess / {len(set(y[v]))} users"
        for k, v in out.items()))
    for a in out:
        for b in out:
            if a < b:
                assert not (set(sess[out[a]]) & set(sess[out[b]])), f"session leak {a}/{b}"
    return out

---
## 8.1 Encoder and loss

`batch_hard_triplet` picks, for every anchor in the batch, the *hardest* positive
(same user, furthest away) and the *hardest* negative (different user, closest),
and pushes them apart. Setting `margin=0` switches to the soft-margin form
`log1p(exp(dp - dn))` — no threshold to tune, and it keeps producing gradient after
the easy triplets are exhausted. Worth trying if training plateaus early.

The distance is Euclidean on L2-normalised vectors, which is a monotone function of
cosine, so the metric used in training is the one used at scoring time.

In [ ]:
class AttnPool(nn.Module):
    """Learned-query attention pooling over the stroke axis."""
    def __init__(self, d):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))

    def forward(self, h):                       # h: (B, N, d)
        w = torch.softmax(self.score(h).squeeze(-1), dim=1)
        return (h * w.unsqueeze(-1)).sum(1), w


class StrokeSetEncoder(nn.Module):
    """window of N stroke-feature vectors -> one L2-normalised embedding.

    Per-stroke MLP, then pooling over the stroke axis. Pooling is permutation
    invariant by design: within a 25-stroke window the ordering is close to
    arbitrary, and an order-sensitive encoder (GRU/transformer) mostly learns
    session-specific sequencing, which is exactly the thing that does not
    transfer to a new session.
    """
    def __init__(self, d_in, mcfg: ModelConfig):
        super().__init__()
        d, p = mcfg.d_hidden, mcfg.dropout
        layers, prev = [], d_in
        for _ in range(mcfg.n_layers):
            layers += [nn.Linear(prev, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p)]
            prev = d
        self.stroke = nn.Sequential(*layers)
        self.pool_mode = mcfg.pool
        self.attn = AttnPool(d) if "attn" in mcfg.pool else None
        mult = {"mean": 1, "meanstd": 2, "attn+meanstd": 3}[mcfg.pool]
        self.head = nn.Sequential(
            nn.Linear(d * mult, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p),
            nn.Linear(d, mcfg.d_embed),
        )

    def forward(self, x, return_attn=False):     # x: (B, N, D)
        h = self.stroke(x)
        parts = [h.mean(1)]
        if "std" in self.pool_mode:
            parts.append(h.std(1, unbiased=False))
        w = None
        if self.attn is not None:
            a, w = self.attn(h)
            parts.insert(0, a)
        z = F.normalize(self.head(torch.cat(parts, dim=-1)), dim=-1)
        return (z, w) if return_attn else z


def pdist(z):
    """Euclidean distance on L2-normalised embeddings == sqrt(2-2cos)."""
    return torch.cdist(z, z, p=2).clamp_min(0)


def batch_hard_triplet(z, labels, margin=0.25):
    """For each anchor: hardest positive, hardest negative, inside the batch.

    margin <= 0 switches to the soft-margin form log1p(exp(dp - dn)), which has
    no threshold to tune and does not go dead once easy triplets are exhausted.
    """
    D = pdist(z)
    same = labels[:, None] == labels[None, :]
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    pos = same & ~eye
    neg = ~same
    valid = pos.any(1) & neg.any(1)
    if not valid.any():
        return z.sum() * 0.0, {}
    dp = (D.masked_fill(~pos, -1e9)).max(1).values[valid]      # hardest positive
    dn = (D.masked_fill(~neg, 1e9)).min(1).values[valid]       # hardest negative
    loss = F.relu(dp - dn + margin).mean() if margin > 0 else F.softplus(dp - dn).mean()
    return loss, {"dp": dp.mean().item(), "dn": dn.mean().item(),
                  "viol": (dp + margin > dn).float().mean().item()}


def contrastive_pairs(z, labels, margin=1.0):
    """Classic siamese pair loss, kept for comparison against batch_hard."""
    D = pdist(z)
    same = (labels[:, None] == labels[None, :]).float()
    iu = torch.triu_indices(len(z), len(z), offset=1, device=z.device)
    d, s = D[iu[0], iu[1]], same[iu[0], iu[1]]
    loss = (s * d.pow(2) + (1 - s) * F.relu(margin - d).pow(2)).mean()
    return loss, {"d_pos": d[s > 0].mean().item(), "d_neg": d[s == 0].mean().item()}


class PKSampler:
    """P users x K windows per batch -- without it, random batches rarely contain
    a positive pair and the mining above has nothing to mine."""
    def __init__(self, y, sess, P, K, steps, seed=0):
        self.rng = np.random.default_rng(seed)
        self.P, self.K, self.steps = P, K, steps
        self.by_user = {u: np.where(y == u)[0] for u in np.unique(y)}
        self.sess = sess
        self.users = [u for u, v in self.by_user.items() if len(v) >= 2]

    def __iter__(self):
        for _ in range(self.steps):
            P = min(self.P, len(self.users))
            picked = self.rng.choice(self.users, size=P, replace=False)
            batch = []
            for u in picked:
                pool = self.by_user[u]
                # spread the K windows over distinct sessions where possible, so the
                # "hardest positive" is a cross-session pair rather than a neighbour
                sess_u = self.sess[pool]
                order = self.rng.permutation(len(pool))
                seen, first, rest = set(), [], []
                for i in order:
                    (first if sess_u[i] not in seen else rest).append(pool[i])
                    seen.add(sess_u[i])
                take = (first + rest)[: self.K]
                if len(take) < self.K:
                    take = list(self.rng.choice(pool, size=self.K, replace=True))
                batch += take
            yield np.array(batch)

    def __len__(self):
        return self.steps

---
## 8.2 Metrics

Three numbers, measuring three different questions:

- **rank-1** (`template_metrics`) — closed-set identification: of the enrolled users,
  who is this? Enrolment comes from train sessions, probes from held-out sessions,
  so it is cross-session by construction.
- **EER** — verification: at what error rate do false accepts equal false rejects?
  This is the number the Balabit literature reports and the one that matters for an
  authentication system.
- **session-level** — average all windows of a session into one embedding first.
  A real system sees a whole session, not 25 strokes, and this is always far better
  than the per-window figure. Quote both; quoting only the session number is how
  papers get to "99.9%".

`pair_metrics` restricts genuine pairs to **different sessions**. Two windows from
the same session are minutes apart, share a posture, a mouse and a mood, and will
match on things that have nothing to do with identity. Including them inflates every
figure and is the single most common way these results get overstated.

In [ ]:
from sklearn.metrics import roc_curve, auc as _auc


def eer_from_scores(y_true, score):
    """score = similarity (higher == more likely genuine)."""
    fpr, tpr, thr = roc_curve(y_true, score)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2), float(_auc(fpr, tpr)), float(thr[i])


@torch.no_grad()
def embed(enc, X, idx=None, bs=512):
    enc.eval()
    A = X if idx is None else X[idx]
    out = [enc(torch.from_numpy(A[i:i + bs]).to(DEVICE)).cpu().numpy()
           for i in range(0, len(A), bs)]
    return np.concatenate(out) if out else np.zeros((0, 1), np.float32)


def templates(z, y):
    """One L2-normalised centroid per user (the 'enrolment' step)."""
    us = np.array(sorted(set(y)))
    T = np.stack([z[y == u].mean(0) for u in us])
    return T / np.linalg.norm(T, axis=1, keepdims=True), us


def template_metrics(z_enroll, y_enroll, z_probe, y_probe, tag=""):
    """Probe each window against every user's enrolment template.

    This is the protocol that matters: enrolment and probe come from different
    sessions by construction, so it cannot be gamed by within-session similarity.
    """
    T, us = templates(z_enroll, y_enroll)
    S = z_probe @ T.T                                   # cosine similarity
    keep = np.isin(y_probe, us)
    S, yp = S[keep], y_probe[keep]
    gt = np.array([list(us).index(u) for u in yp])
    rank1 = float((S.argmax(1) == gt).mean())
    lab = np.zeros_like(S, dtype=int); lab[np.arange(len(gt)), gt] = 1
    eer, auc_, thr = eer_from_scores(lab.ravel(), S.ravel())
    m = {"rank1": rank1, "eer": eer, "auc": auc_, "thr": thr, "n_probe": int(len(yp))}
    if tag:
        print(f"[{tag}] rank-1 {rank1:6.1%}   EER {eer:6.2%}   AUC {auc_:.4f}   "
              f"(n={len(yp)}, {len(us)} users)")
    return m


def pair_metrics(z, y, sess, cross_session_only=True, max_pairs=200_000, seed=0, tag=""):
    """Window-vs-window verification.

    Genuine pairs are restricted to DIFFERENT sessions. Same-session pairs are
    minutes apart at most and inflate every number you would report.
    """
    rng = np.random.default_rng(seed)
    n = len(z)
    iu = np.triu_indices(n, 1)
    if len(iu[0]) > max_pairs:
        sel = rng.choice(len(iu[0]), max_pairs, replace=False)
        iu = (iu[0][sel], iu[1][sel])
    i, j = iu
    same_u, same_s = y[i] == y[j], sess[i] == sess[j]
    keep = ~same_s if cross_session_only else np.ones(len(i), bool)
    s = (z[i] * z[j]).sum(1)[keep]
    lab = same_u[keep].astype(int)
    if lab.sum() == 0 or (1 - lab).sum() == 0:
        return {"eer": np.nan, "auc": np.nan, "n_pairs": int(keep.sum())}
    eer, auc_, thr = eer_from_scores(lab, s)
    m = {"eer": eer, "auc": auc_, "thr": thr, "n_pairs": int(keep.sum()),
         "n_genuine": int(lab.sum())}
    if tag:
        print(f"[{tag}] EER {eer:6.2%}   AUC {auc_:.4f}   "
              f"({lab.sum():,} genuine / {(1-lab).sum():,} impostor pairs)")
    return m


def session_level(z, y, sess, z_enroll, y_enroll, tag=""):
    """Average a session's windows into one embedding before deciding.

    A real system gets a whole session, not one 25-stroke window; this is the
    number to quote for 'can we tell who is at the keyboard', and it is always
    much better than the per-window number.
    """
    keys = sorted(set(zip(y, sess)))
    Z = np.stack([z[(y == u) & (sess == s)].mean(0) for u, s in keys])
    Z /= np.linalg.norm(Z, axis=1, keepdims=True)
    yy = np.array([u for u, _ in keys])
    return template_metrics(z_enroll, y_enroll, Z, yy, tag=tag)


import matplotlib.pyplot as plt


def plot_embeddings(z, y, title="test embeddings (PCA)"):
    zc = z - z.mean(0)
    U, S, Vt = np.linalg.svd(zc, full_matrices=False)
    P = zc @ Vt[:2].T
    fig, ax = plt.subplots(figsize=(6, 5))
    for u in sorted(set(y)):
        m = y == u
        ax.scatter(P[m, 0], P[m, 1], s=14, alpha=.75, label=u)
    ax.set_title(f"{title}  ({S[:2].sum()/S.sum():.0%} of variance)")
    ax.legend(fontsize=7, ncol=2, markerscale=1.5)
    plt.tight_layout(); plt.show()

---
## 8.3 Train

Early stopping on validation EER, not on loss — triplet loss goes to ~0 once the
batch stops containing violations, which says nothing about whether the embedding
generalises.

Watch `dp` and `dn` in the log. If `dn` climbs while `dp` stays flat, it is
separating users properly. If both collapse toward 0 the embedding is degenerating
to a constant; drop the learning rate or use `margin=0`.

In [ ]:
def train_siamese(X, y, sess, idx, mcfg: ModelConfig, verbose_every=5):
    torch.manual_seed(mcfg.seed); np.random.seed(mcfg.seed)
    tr, va = idx["train"], idx.get("val", idx["train"])

    enc = StrokeSetEncoder(X.shape[2], mcfg).to(DEVICE)
    opt = torch.optim.AdamW(enc.parameters(), lr=mcfg.lr, weight_decay=mcfg.weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=mcfg.lr, total_steps=mcfg.epochs * mcfg.steps_per_epoch, pct_start=0.2)

    Xtr, ytr, str_ = X[tr], y[tr], sess[tr]
    sampler = PKSampler(ytr, str_, mcfg.P_users, mcfg.K_windows, mcfg.steps_per_epoch, mcfg.seed)
    lut = {u: i for i, u in enumerate(sorted(set(ytr)))}

    best, best_state, bad, hist = np.inf, None, 0, []
    for ep in range(1, mcfg.epochs + 1):
        enc.train(); tot, stats = 0.0, {}
        for b in sampler:
            xb = torch.from_numpy(Xtr[b]).to(DEVICE)
            lb = torch.as_tensor([lut[u] for u in ytr[b]], device=DEVICE)
            z = enc(xb)
            if mcfg.loss == "batch_hard":
                loss, st = batch_hard_triplet(z, lb, mcfg.margin)
            else:
                loss, st = contrastive_pairs(z, lb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(enc.parameters(), 5.0)
            opt.step(); sched.step()
            tot += loss.item(); stats = st

        z_tr, z_va = embed(enc, X, tr), embed(enc, X, va)
        m = template_metrics(z_tr, ytr, z_va, y[va])
        hist.append({"epoch": ep, "loss": tot / mcfg.steps_per_epoch,
                     "val_eer": m["eer"], "val_rank1": m["rank1"], **stats})
        if verbose_every and (ep % verbose_every == 0 or ep == 1):
            print(f"  ep {ep:3d}  loss {hist[-1]['loss']:.4f}  "
                  f"val EER {m['eer']:6.2%}  val rank-1 {m['rank1']:6.1%}"
                  + (f"  dp {stats['dp']:.3f} dn {stats['dn']:.3f}" if "dp" in stats else ""))

        if m["eer"] < best - 1e-4:
            best, bad = m["eer"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in enc.state_dict().items()}
        else:
            bad += 1
            if bad >= mcfg.patience:
                print(f"  early stop at epoch {ep} (best val EER {best:.2%})")
                break

    if best_state is not None:
        enc.load_state_dict(best_state)
    print(f"[train] best val EER {best:.2%}")
    return enc, pd.DataFrame(hist)


def evaluate(enc, X, y, sess, idx, mcfg: ModelConfig):
    tr = idx["train"]
    z_tr = embed(enc, X, tr)
    out = {}
    for fold in ("val", "test"):
        if fold not in idx:
            continue
        f = idx[fold]
        z = embed(enc, X, f)
        out[f"{fold}_window"]  = template_metrics(z_tr, y[tr], z, y[f], tag=f"{fold} window")
        out[f"{fold}_session"] = session_level(z, y[f], sess[f], z_tr, y[tr], tag=f"{fold} session")
        out[f"{fold}_pairs"]   = pair_metrics(z, y[f], sess[f], tag=f"{fold} pairs  ")
    if "unseen" in idx:
        u = idx["unseen"]
        zu = embed(enc, X, u)
        print("\n-- users never seen in training (open-set: the honest number) --")
        out["unseen_pairs"] = pair_metrics(zu, y[u], sess[u], tag="unseen pairs")
    return out


mcfg = ModelConfig(n_strokes=25, stride=8, d_embed=64, loss="batch_hard", margin=0.25,
                   P_users=8, K_windows=6, epochs=60, steps_per_epoch=60, seed=0)

# Hold out 2 users entirely -> the open-set test further down. Set to () to train on all.
HOLDOUT_USERS = ()

st_f  = assign_folds(strokes, val_frac=0.15, test_frac=0.20, seed=0,
                     holdout_users=HOLDOUT_USERS)
pre_m = fit_preprocessor(st_f[st_f.fold == "train"], PREP)   # fitted on the train fold only
X, y, sess, folds = build_windows(st_f, pre_m, mcfg)
idx = fold_index(folds, sess, y)

enc, hist = train_siamese(X, y, sess, idx, mcfg)

---
## 8.4 Evaluate

If `HOLDOUT_USERS` is non-empty, the last block is the honest number: users the
model has never seen, scored purely on whether the embedding puts *any* two windows
from the same stranger closer together than windows from different strangers. That
is what an embedding model is for, and it is normally a few points worse than the
seen-user figures. With only 10 Balabit users, holding 2 out costs real training
signal — worth running once to get the number, then training on all 10 for the
deployed model.

In [ ]:
res = evaluate(enc, X, y, sess, idx, mcfg)

z_test = embed(enc, X, idx["test"])
plot_embeddings(z_test, y[idx["test"]])

display(hist.tail(10))

---
## 8.5 Save, and score a new session

`score_session()` closes the loop: raw CSV → clean → strokes → features →
preprocess → windows → embedding → similarity against every enrolled user. Same
code path as training, which is the point of saving the preprocessor alongside the
weights.

For an authentication decision rather than a ranking, use the threshold in
`res["test_session"]["thr"]` — the EER operating point. Move it up for fewer false
accepts, down for fewer false rejects; there is no symmetric choice that is
automatically right.

In [ ]:
def save_model(outdir, enc, pre, mcfg, metrics, hist, z_enroll, y_enroll, tag="siamese_v1"):
    out = Path(outdir) / tag
    out.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": enc.state_dict(), "d_in": enc.stroke[0].in_features,
                "model_config": asdict(mcfg), "features": pre["features"]},
               out / "encoder.pt")
    T, us = templates(z_enroll, y_enroll)
    np.savez(out / "templates.npz", T=T, users=us)
    pd.DataFrame(hist).to_csv(out / "history.csv", index=False)
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))
    joblib.dump(pre, out / "preprocessor.joblib")
    print(f"[save] -> {out}  ({len(us)} enrolled users, {T.shape[1]}-d embeddings)")
    return out


def load_model(outdir, tag="siamese_v1"):
    out = Path(outdir) / tag
    ck = torch.load(out / "encoder.pt", map_location=DEVICE, weights_only=False)
    mcfg = ModelConfig(**ck["model_config"])
    enc = StrokeSetEncoder(ck["d_in"], mcfg).to(DEVICE)
    enc.load_state_dict(ck["state_dict"]); enc.eval()
    t = np.load(out / "templates.npz", allow_pickle=True)
    return enc, mcfg, joblib.load(out / "preprocessor.joblib"), t["T"], t["users"]


def score_session(raw_csv_or_df, enc, pre, mcfg, T, users,
                  cfg: Config = Config(), pcfg: PrepConfig = None):
    """Raw session -> one embedding -> similarity to every enrolled user.

    Runs the same clean -> strokes -> features -> preprocess chain as training.
    Returns (ranking DataFrame, session embedding, n_windows).
    """
    pcfg = pcfg or PREP
    raw = pd.read_csv(raw_csv_or_df) if isinstance(raw_csv_or_df, (str, Path)) else raw_csv_or_df
    st, qc = process_frame(raw, "probe", "probe", replace(cfg, verbose=False), pcfg)
    if not len(st):
        raise RuntimeError(f"session rejected by QC: {qc.get('drop_reason')}")
    st = filter_strokes(st, pcfg)
    st["fold"] = "probe"
    Xp, _, _, _ = build_windows(st, pre, mcfg)
    z = embed(enc, Xp)
    zs = z.mean(0); zs /= np.linalg.norm(zs)
    rank = (pd.DataFrame({"user": users, "similarity": T @ zs})
              .sort_values("similarity", ascending=False).reset_index(drop=True))
    return rank, zs, len(z)


MODEL_ROOT = ROOT / "models"            # -> ./data/models

z_enroll = embed(enc, X, idx["train"])
save_model(MODEL_ROOT, enc, pre_m, mcfg, res, hist, z_enroll, y[idx["train"]], tag="siamese_v1")

# --- later / elsewhere -------------------------------------------------------
# enc2, mcfg2, pre2, T, users = load_model(MODEL_ROOT, "siamese_v1")
# rank, z_sess, n_win = score_session(some_csv_path, enc2, pre2, mcfg2, T, users)
# display(rank.head())